# BLIP-2 Caption Generation — Offline Indexing
**Role in pipeline:** Step 2 of offline indexing.  
For every catalog image, BLIP-2 generates a text caption describing the clothing item (color, fit, material, style). These captions are later used by CLIP's text encoder to build fused embeddings.

> **BLIP-2 stays frozen** — no fine-tuning needed. We only run inference.

**Cropping strategy (Option 2):**
- **Offline indexing** → use bbox annotations from `list_bbox_inshop.txt` directly (ground truth crops, faster, no model inference needed)
- **Online inference** → use fine-tuned YOLO to crop the query image at runtime

**Inputs:** Raw catalog images + bbox annotations  
**Outputs:** `captions.json` → `{ "image_name": "caption text", ... }`


## Cell 1 — Install Dependencies

In [1]:
!pip install transformers accelerate Pillow tqdm --quiet
!pip install ultralytics --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.1 MB/s eta 0:00:00


## Cell 2 — Imports & Paths

In [2]:
import json
import torch
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from transformers import Blip2Processor, Blip2ForConditionalGeneration

# ── Paths (your working Kaggle paths) ──
DATASET_ROOT  = Path("/kaggle/input/datasets/ashok1145/vr-fproj/vr_final_proj_dataset")
IMG_ROOT      = DATASET_ROOT / "img" / "img"
SPLIT_FILE    = DATASET_ROOT / "eval" / "list_eval_partition.txt"
BBOX_FILE     = DATASET_ROOT / "Anno" / "list_bbox_inshop.txt"

OUTPUT_DIR    = Path("/kaggle/working")
CAPTIONS_FILE = OUTPUT_DIR / "captions.json"

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 8      # reduce to 4 if OOM on T4
PAD        = 0.05   # padding fraction around bbox crop

print(f"Using device: {DEVICE}")
print(f"IMG_ROOT exists: {IMG_ROOT.exists()}")


Using device: cuda
IMG_ROOT exists: True


## Cell 3 — Load BLIP-2
We use `Salesforce/blip2-opt-2.7b` — the smaller OPT variant, fits comfortably on T4.  
YOLO is **not loaded here** — offline indexing uses bbox annotations directly.


In [3]:
BLIP2_MODEL = "Salesforce/blip2-opt-2.7b"
processor   = Blip2Processor.from_pretrained(BLIP2_MODEL, use_fast=True)
blip2_model = Blip2ForConditionalGeneration.from_pretrained(
    BLIP2_MODEL,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto",
)
blip2_model.eval()
print("BLIP-2 loaded.")


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/882 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/accelerate/utils/modeling.py:1598: UserWarning: The following device_map keys do not match any submodules in the model: ['query_tokens']
  warnings.warn(


BLIP-2 loaded.


## Cell 4 — Parse Annotation Files
Read image paths, item IDs, split assignments, and **ground truth bboxes**.  
We caption **train + gallery** images — these form the offline index.  
Query images are cropped at runtime by YOLO during online retrieval.


In [4]:
def parse_split_file(path):
    with open(path) as f:
        lines = f.readlines()
    rows = []
    for line in lines[2:]:
        parts = line.strip().split()
        if len(parts) < 3:
            continue
        raw = parts[0]
        img_name = raw[len("img/"):] if raw.startswith("img/") else raw
        rows.append({"image_name": img_name, "item_id": parts[1], "split": parts[2]})
    return rows


def parse_bbox_file(path):
    """
    Returns dict: image_name -> (x1, y1, x2, y2) absolute pixel coords.
    Used for offline indexing — ground truth crops, no YOLO inference needed.
    """
    with open(path) as f:
        lines = f.readlines()
    bboxes = {}
    for line in lines[2:]:
        parts = line.strip().split()
        if len(parts) < 7:
            continue
        raw = parts[0]
        img_name = raw[len("img/"):] if raw.startswith("img/") else raw
        bboxes[img_name] = (int(parts[3]), int(parts[4]), int(parts[5]), int(parts[6]))
    return bboxes


all_rows = parse_split_file(SPLIT_FILE)
bbox_map = parse_bbox_file(BBOX_FILE)

# Caption train + gallery only
index_images = [r for r in all_rows if r["split"] in ("train", "gallery")]

print(f"Total images in split file : {len(all_rows)}")
print(f"Train + gallery to caption : {len(index_images)}")
print(f"BBox annotations loaded    : {len(bbox_map)}")


Total images in split file : 52712
Train + gallery to caption : 38494
BBox annotations loaded    : 52712


## Cell 5 — BBox Crop Helper (Offline)
For offline indexing we crop using the **ground truth bbox annotations** — no YOLO inference.  
This is faster, deterministic, and gives perfect crops since annotations are already verified.

> For online query images (at retrieval time), the fine-tuned YOLO crops instead.


In [5]:
def bbox_crop(img_path: Path, bbox: tuple, pad: float = PAD) -> Image.Image:
    """
    Crop using ground truth bbox annotation.
    bbox: (x1, y1, x2, y2) absolute pixel coords from list_bbox_inshop.txt
    pad : fraction of box size to add as padding on each side
    """
    pil_img = Image.open(img_path).convert("RGB")
    W, H    = pil_img.size
    x1, y1, x2, y2 = bbox

    # Add padding
    px = int((x2 - x1) * pad)
    py = int((y2 - y1) * pad)
    x1 = max(0, x1 - px)
    y1 = max(0, y1 - py)
    x2 = min(W, x2 + px)
    y2 = min(H, y2 + py)

    # Sanity check — skip degenerate boxes
    if x2 <= x1 or y2 <= y1:
        return pil_img   # fallback: full image

    return pil_img.crop((x1, y1, x2, y2))


## Cell 6 — Generate Captions
Run BLIP-2 over all train + gallery images in batches.

**Prompt:** *"Question: Describe this clothing item including color, style, fit, and material. Answer:"*  
This steers BLIP-2 toward product-relevant descriptions rather than generic scene captions.

Crops are made using **ground truth bboxes** (offline indexing strategy).  
Images with no bbox annotation fall back to the full image.


In [6]:
PROMPT   = "Question: Describe this clothing item including color, style, fit, and material. Answer:"
captions = {}
skipped  = 0

for i in tqdm(range(0, len(index_images), BATCH_SIZE), desc="Captioning"):
    batch = index_images[i : i + BATCH_SIZE]
    crops, names = [], []

    for row in batch:
        img_path = IMG_ROOT / row["image_name"]
        if not img_path.exists():
            skipped += 1
            continue
        try:
            bbox = bbox_map.get(row["image_name"])
            if bbox:
                crop = bbox_crop(img_path, bbox)   # ← ground truth crop (offline)
            else:
                crop = Image.open(img_path).convert("RGB")  # fallback: no bbox
            crops.append(crop)
            names.append(row["image_name"])
        except Exception as e:
            skipped += 1
            continue

    if not crops:
        continue

    inputs = processor(
        images=crops,
        text=[PROMPT] * len(crops),
        return_tensors="pt",
        padding=True,
    ).to(DEVICE, torch.float16 if DEVICE == "cuda" else torch.float32)

    with torch.no_grad():
        generated_ids = blip2_model.generate(
            **inputs,
            max_new_tokens=60,
            num_beams=3,
        )

    texts = processor.batch_decode(generated_ids, skip_special_tokens=True)

    for name, text in zip(names, texts):
        caption = text.replace(PROMPT, "").strip()
        captions[name] = caption

print(f"\nCaptioned : {len(captions)}")
print(f"Skipped   : {skipped}")
print("\nSample captions:")
for k, v in list(captions.items())[:3]:
    print(f"  {k}\n    → {v}\n")



Captioning:   0%|          | 0/4812 [00:00<?, ?it/s]The `language_model` is not in the `hf_device_map` dictionary and you are running your script in a multi-GPU environment. this may lead to unexpected behavior when using `accelerate`. Please pass a `device_map` that contains `language_model` to remove this warning. Please refer to https://github.com/huggingface/blog/blob/main/accelerate-large-models.md for more details on creating a `device_map` for large models.

Captioning:   0%|          | 1/4812 [00:05<6:55:42,  5.18s/it]The `language_model` is not in the `hf_device_map` dictionary and you are running your script in a multi-GPU environment. this may lead to unexpected behavior when using `accelerate`. Please pass a `device_map` that contains `language_model` to remove this warning. Please refer to https://github.com/huggingface/blog/blob/main/accelerate-large-models.md for more details on creating a `device_map` for large models.

Captioning:   0%|          | 2/4812 [00:08<5:10:4


Captioned : 38494
Skipped   : 0

Sample captions:
  WOMEN/Dresses/id_00000002/02_1_front.jpg
    → Floral print dress

  WOMEN/Dresses/id_00000002/02_2_side.jpg
    → Black floral print mini dress

  WOMEN/Dresses/id_00000002/02_4_full.jpg
    → black floral print mini dress



## Cell 7 — Save Captions
Save `captions.json` to the Kaggle output tab.  
Upload this file as a Kaggle dataset before running the CLIP ablation notebook.


In [7]:
with open(CAPTIONS_FILE, "w") as f:
    json.dump(captions, f, indent=2)

print(f"Saved {len(captions)} captions → {CAPTIONS_FILE}")
print("\nDownload from the Kaggle output tab and upload as a dataset for CLIP training.")


Saved 38494 captions → /kaggle/working/captions.json

Download from the Kaggle output tab and upload as a dataset for CLIP training.


## Note — Online Query Cropping (at Retrieval Time)
<!-- At retrieval time (Streamlit demo / batch eval), query images are **not** in the annotation file.  
For those, the fine-tuned YOLO model handles cropping: -->

```python
from ultralytics import YOLO
yolo = YOLO("/kaggle/input/models/taralsanka/best-yolo-8l-upd/other/default/1/best_yolo8l.pt")

def online_crop(img_path, pad=0.05):
    pil_img = Image.open(img_path).convert("RGB")
    result  = yolo.predict(source=pil_img, conf=0.25, verbose=False)[0]
    if result.boxes and len(result.boxes):
        best = max(result.boxes, key=lambda b: float(b.conf[0]))
        x1, y1, x2, y2 = best.xyxy[0].cpu().numpy().astype(int)
        W, H = pil_img.size
        px = int((x2-x1)*pad); py = int((y2-y1)*pad)
        return pil_img.crop((max(0,x1-px), max(0,y1-py), min(W,x2+px), min(H,y2+py)))
    return pil_img   # fallback
```
